In [1]:
## This is a 3-part document, made from three separate files. It is made to generate a single graph at the end.

# From Strain rate code
# To handle images, we have PIL - Image
#from PIL import Image # PIL might be defunct 
import numpy as np  # Essential functions
import pandas as pd # Essential for dataframe and excel manipulation
import matplotlib.pyplot as plt # Essential data visualization

# Collection of image processing directories.
#from skimage.filters import threshold_otsu #The binarization algorithm
from skimage.io import imsave#, imread #Image processing
#from skimage.morphology import reconstruction # For floodfill algorithm
%matplotlib inline 
# automatic production of plots made in program
#from sklearn.linear_model import LinearRegression # function that generates a linear fit. Might be defunct
from mpl_toolkits.axes_grid1 import make_axes_locatable # might be defunct.
import os as os # os for handling working folder
import os.path as ospath # os.path used to check if the csv exists.
from scipy.optimize import curve_fit # function that generates a linear fit. Might be defunct



In [2]:
# Essential path names and opening terms

# This block assigns the directory we work in, and the filenames we generate and use.

# Specify the overarching path for our filenames.
path = "C:\\Users\\franc\\Box\\ALAB Data Francis Cavanna\\2021 Actomyosin Control\\"

# Specify base of the filenames of the series of images:
base = 'StrainRateFiles\\Actin_12uM_Fascin_1,2uM_Myosin_0,12uM-Create Image Subset and Split-01'

# Specify suffix of the filenames of the series of images
suffix = '.tif'

# Specify originator Excel Document from MolarConcentration.nb program:
MCfilename = "MathematicaDF.csv"

# Specify truncation term:
truncate = 0

# Specify number of slices
N_slices = 91

# Set fontsize, used when generating plots
ftsz = 20
tcksz = 14

# Set length of indexes
index = 88  #I've actually moved this into each individual function. Index must be 88 for energy terms 
#index = 84

In [3]:
#Start function for calculating strain here:
def Calc_StrainRates(images,filenames,folder):

    #Create list to hold pixel areas.
    areas = []

    #Create list of filenames so we can pull out the images and process them.
    for i in range(0,len(images)):
        fill = images[i]
        fn = filenames[i]

        imsave(fn + ".jpeg",fill)   # This causes the "Lossy conversion error". To fix, it's possible to convert the float64
        # numerical type to a uint8 number type. I'm not going to do this right now.

        # Sum the number of pixels in the image to get an area. This will be used to calculate Hencky Strain.
        area = sum(sum(fill))
        areas.append(area)

    # Square the total area covered by pixels to get a length measurement. This is used to calculate Hencky Strain.
    lengths = np.asarray(areas)**(1/2)
    #print(areas[15])
    #Once we process these iamges, we need to get a timescale. 
    
    #The number of frames of data is 91. The number of measurements of Hencky Strain we get is 89. 
    # Hencky Strain is calculated by a difference of length scales; the maximum we can get in infitesimal steps is 90.
    # To get time derivative in hencky strain we must take Euler's approximation for the derivative. So that's 89 measurements.
    # Time derivative in Hencky strain is also centered on 20s to 1780s datapoints.
    frames = np.arange(0,N_slices-2)
    times = frames * 20

    henckyStrainDerivative = []
    sum_strain = []
    sum_strain_value = 0.0
    times = []

    # We now calculate the hencky strain using Euler's algorithm. 
    for i in range(0, len(lengths)-1):
        # Pull difference in lengths to get incremental change in strain.
        dl = lengths[i+1] - lengths[i]
        # Calculate square root difference in areas over a timestep for incremental change in area squared.
        Diff = -np.sqrt(np.absolute(areas[i] - areas[i+1]))

        # Calculate time derivative of hencky Strain
        henckyStrainDerivative.append(dl/lengths[i]/20.0)

        #Calculate Hencky Strain at each point in time.
        if i >= truncate:
            sum_strain_value = sum_strain_value + dl/lengths[i]
            sum_strain.append(sum_strain_value)

        times.append((i+1)*20.0)

    # Pull out the highest strain and rescale the strain so it plots nicely
    a = np.amax(sum_strain)

    for i in range(0,len(sum_strain)):
        sum_strain[i] = sum_strain[i] - a

    # We now save the data in the format "Total Strain" (unitless), "infinitesimal Strain/time" (s^-1), "time" (s)
    Data = [sum_strain,henckyStrainDerivative,times]
    
    # Take care of previous strains.csv file which may or may not exist.
    if ospath.exists(path + folder + '\Results//Strains.csv') is True:
        # Delete the previous Strains.csv
        os.remove(path + folder + '\Results//Strains.csv')
    else:
        pass
    # Turn into a dataframe to save the file.
    df = pd.DataFrame(Data,index=["Strain","Strain_dot","Time (s)"])
    # save to collect strain rates from the csv.
    df.to_csv(path + folder + '\Results//Strains.csv', index=True, mode='a',header=False)
    
    pixel_length = 0.7692 #microns per pixel
    height = 152.9 #microns
    Length_Data = [lengths,pixel_length*lengths,
                   (pixel_length*lengths/4+height*pixel_length*lengths/np.sqrt(areas[0])/pixel_length),
                                        times]
    
    #df2 = pd.DataFrame(Length_Data,index=['pixels','length (um)','Volume/SurfaceAr','Time'])
    #df2.to_csv(path + folder + '\Results//MattiaRep.csv', index=True, mode='a',header=False)



In [4]:
def Create_Strain_Plot(path,folder):
    index = 88
    if folder == '2022_3_15_Anthony//1,2 F-0,24 M-2':
        index = 59
        # This block of code is for the one dataset that has secondary contraction
        Strains = pd.read_csv(path + folder + '\Results\Strains.csv',header=None)
        Energies = pd.read_csv(path + folder + '\Results\Energies.csv',header=None)

        # Make a list of strain measurements and energy measurements
        sum_strain = np.absolute(list(Strains.iloc[0]))
        sum_energy = list(Energies.iloc[0])
        times = Strains.iloc[2]
        #Plot cumulative strain vs. cumulative Davolio energy consumption. Truncate before secondary contraction occurs
        temperature = times[0:59]

        # make colors for graph to understand exactly which measurements are there.
        colors = temperature
        # Make a plot of the energy vs the strain. Add labels as needed.
        print("Length of strain", len(sum_strain[0:index]))
        print("Length of energy", len(sum_energy[0:index]))
        print("Length of times", len(colors[0:index]))
        plot = plt.scatter(sum_energy[0:59],sum_strain[0:59],c=colors)
        plt.title('Cumulative Strain vs Rescaled Cumulative Energy Consumption')
        # We always plot in units of J/L.
        plt.xlabel('$e/V \; (J/L)$',fontsize=20)
        plt.ylabel(r'$\epsilon$',fontsize=20)
        plt.tick_params(axis='both', which='major', labelsize=tcksz)
        plt.xscale('log')
        plt.yscale('log')

        # Add a colorbar to track time throughout the plot
        cb = plt.colorbar(plot)

        # Save the plot
        plt.savefig(path+ folder + '\Results//LogLog_HS_eC.png',dpi="figure",format="png",bbox_inches='tight')
        # You have to remove colobars or they successively add to the plot window if you batch process a bunch of folders
        cb.remove()
        plt.clf()

        #plt.show()
    
    else:
        Strains = pd.read_csv(path + folder + '\Results\Strains.csv',header=None)
        Energies = pd.read_csv(path + folder + '\Results\Energies.csv',header=None)

        # Make a list of strain measurements and energy measurements
        sum_strain = np.absolute(list(Strains.iloc[0]))
        sum_energy = list(Energies.iloc[0])
        times = Strains.iloc[2]
        #Plot cumulative strain vs. cumulative Davolio energy consumption
        temperature = times[0:index]


        # make colors for graph to understand exactly which measurements are there.
        colors = temperature 
        # Make a plot of the energy vs the strain. Add labels as needed.
        #print("Length of strain", len(sum_strain[0:index]))
        #print("Length of energy", len(sum_energy[0:index]))
        #print("Length of strain", len(times[0:index]))
        plot = plt.scatter(sum_energy[0:index],sum_strain[0:index],c=colors)
        plt.title('Cumulative Strain vs Rescaled Cumulative Energy Consumption')
        # We always plot in units of J/L.
        plt.xlabel('$e/V \; (J/L)$',fontsize=20)
        plt.ylabel(r'$\epsilon$',fontsize=20)
        plt.tick_params(axis='both', which='major', labelsize=tcksz)
        plt.xscale('log')
        plt.yscale('log')

        # Add a colorbar to track time throughout the plot
        cb = plt.colorbar(plot)

        # Save the plot
        plt.savefig(path+ folder + '\Results//LogLog_HS_eC.png',dpi="figure",format="png",bbox_inches='tight')
        # You have to remove colobars or they successively add to the plot window if you batch process a bunch of folders
        cb.remove()
        plt.clf()

        #plt.show()

In [5]:
# Python Curve Fit

# This code plots data on a logarithmic scale, and tries to fit it to a y = m*x + b fit.
def Plot_Logs(Begin,path,folder,sum_strain,Energy):

    StopPoint = 89
    index = 88
    
    # Create the absolute value of strain
    false_strain = np.absolute(sum_strain)

    # Take the log of each datapoint:
    log_falsestrain = []
    log_DavolioEnergy = []
    for i in range(0,len(false_strain)):
        p = np.log(false_strain[i])
        q = np.log(Energy[i])
        log_falsestrain.append(p)
        log_DavolioEnergy.append(q)
    
    # Fit the data to when each experiment begins contraction.
    #print(type(Begin))
    m,b = np.polyfit(log_DavolioEnergy[Begin:StopPoint],log_falsestrain[Begin:StopPoint],1)

    # Print the first-order polynomial fitting coefficients
    print(m,b)

    # Re-create the fitted logarithm as a line.
    log_DavolioStrain2 = []
    for i in range(0,len(log_DavolioEnergy[Begin:index])):
        p = log_DavolioEnergy[i+Begin]*m + b
        log_DavolioStrain2.append(p)

    # Plot the data and the fitted line against each other.:
    plt.scatter(log_DavolioEnergy[Begin:index], log_falsestrain[Begin:index], label = 'Data')
    plt.plot(log_DavolioEnergy[Begin:index],log_DavolioStrain2,'r--',label='$\epsilon$ = '+str(np.exp(b))[0:4] + '*exp{'+str(m)[0:4]+'*$\mathrm{log}$(e/V)}')
    # We always use J/L
    plt.xlabel('$\mathrm{log}(e/V) \; (J/L)$',fontsize=20)
    plt.ylabel(r'$\mathrm{log}(\epsilon)$',fontsize=20)
    plt.tick_params(axis='both', which='major', labelsize=tcksz)
    plt.legend(loc="upper left")
    plt.savefig(path+ folder + '\Results//LogFitting_2.png',dpi="figure",format="png")
    plt.clf()

    # Fit the data before my division in time to y=m*x + b:
    m,b = np.polyfit(log_DavolioEnergy[0:(Begin-1)],log_falsestrain[0:(Begin-1)],1)
    log_DavolioStrain2 = []
    for i in range(0,len(log_DavolioEnergy[0:(Begin-1)])):
        p = log_DavolioEnergy[i]*m + b
        log_DavolioStrain2.append(p)

    # Plot the data and the fitted line before my division in time:
    plt.scatter(log_DavolioEnergy[0:(Begin-1)], log_falsestrain[0:(Begin-1)], label = 'Data')
    plt.plot(log_DavolioEnergy[0:(Begin-1)],log_DavolioStrain2,'r--',label='$\epsilon$ = '+str(np.exp(b))[0:4] + '*exp{'+str(m)[0:4]+'*$\mathrm{log}$(e/V)}')
    plt.xlabel('$\mathrm{log}(e/V) \; (J/mL )$',fontsize=20)
    plt.ylabel(r'$\mathrm{log}(\epsilon)$',fontsize=20)
    plt.tick_params(axis='both', which='major', labelsize=tcksz)
    plt.legend(loc="upper left")
    plt.savefig(path+ folder + '\Results//LogFitting_1.png',dpi="figure",format="png")
    plt.clf()


In [6]:
# Generalized Curve fitting with an exponential function.

def exponential(x,a,b,c,d):
    return a*np.exp(b*x+c)+d

def find_exponential(sum_energies,sum_strains,times,path,folder):

    # Get length of the sum_strains data
    l = len(sum_strains)
    
    # Fit the data according to an exponential power law. p0 gives initial coefficient guess.
    pars, cov = curve_fit(f=exponential, xdata=sum_energies, ydata=sum_strains, p0=[1, -10,0,-1])
    # Return exponential fit paramemters
    print(pars)

    #Create function that plots the line in the curve fitting:
    calc_strains = []
    for i in range(l):
        ev = exponential(sum_energies[i],*pars)
        calc_strains.append(ev)

    # Plot the results from the data and the curve:
    fig = plt.figure()
    ax = fig.add_subplot(111)

    temperature = times[0:l]
    
    colors = temperature 
    # Plot the data
    plt.scatter(sum_energies[0:l],sum_strains[0:l],c=colors)
    # Plot the curve of predicted strain vs energy.
    plt.plot(sum_energies[0:l],calc_strains, label='$\epsilon$ = '+str(pars[0])[0:4] + '*exp{'+str(pars[1])[0:8]+
             '*(e/V) + '+str(pars[2])[0:4]+'}'+str(pars[3])[0:4])
    #plt.title('Cumulative Strain vs Cumulative Energy Consumption')
    # We always plot as J/L
    plt.xlabel('$e \; (J/L)$',fontsize=20)
    plt.ylabel(r'$\epsilon$',fontsize=20)
    plt.tick_params(axis='both', which='major', labelsize=tcksz)
    plt.legend(loc="upper right")
    plt.savefig(path+ folder + '\Results//ExpFitting_2.png',dpi="figure",format="png",bbox_inches='tight')
    plt.clf()
    #plt.xscale('log')
    #plt.yscale('log')

    #plt.colorbar(colors)

    #plt.show()
    
    ### Make a plot for resuiduals: difference between predicted values and datapoints
    difference = []
    for i in range(l):
        difference.append(calc_strains[i]-sum_strains[i])
    plt.plot(sum_energies[0:l],difference)#,c=colors
    plt.xlabel('$g \; (J/L)$',fontsize=20)
    plt.ylabel(r'$\epsilon_{pred}-\epsilon_{data}$',fontsize=20)
    plt.tick_params(axis='both', which='major', labelsize=tcksz)
    plt.savefig(path+ folder + '\Results//ExpFitting_Residuals.png',dpi="figure",format="png",bbox_inches='tight')
    plt.clf()
    
    # Return desirable fitted parameters:
    return pars[1]

In [ ]:
# Generalized Curve fitting with an exponential function. Repeat of last code but with the 
# Absolute value of the strain

def exponential(x,a,b,c,d):
    return a*np.exp(b*x+c)+d

def abs_exponential(sum_energies,sum_strains,times,path,folder):

    # Get length of the sum_strains data
    l = len(sum_strains)
    
    # Fit the data according to an exponential power law. p0 gives initial coefficient guess.
    pars, cov = curve_fit(f=exponential, xdata=sum_energies, ydata=sum_strains, p0=[-1, -0.05,1,1])
    # Return exponential fit paramemters
    print(pars)

    #Create function that plots the line in the curve fitting:
    calc_strains = []
    for i in range(l):
        ev = exponential(sum_energies[i],*pars)
        calc_strains.append(ev)

    # Plot the results from the data and the curve:
    fig = plt.figure()
    ax = fig.add_subplot(111)

    temperature = times[0:l]
    
    colors = temperature 
    # Plot the data
    plt.scatter(sum_energies[0:l],sum_strains[0:l],c=colors)
    # Plot the curve of predicted strain vs energy.
    plt.plot(sum_energies[0:l],calc_strains, label='$\epsilon$ = '+str(pars[0])[0:4] + '*exp{'+str(pars[1])[0:8]+
             '*(e/V) + '+str(pars[2])[0:4]+'}'+str(pars[3])[0:4])
    #plt.title('Cumulative Strain vs Cumulative Energy Consumption')
    # We always plot as J/L
    plt.xlabel('$e \; (J)$',fontsize=20)
    plt.ylabel(r'$\epsilon$',fontsize=20)
    plt.tick_params(axis='both', which='major', labelsize=tcksz)
    plt.legend(loc="lower right")
    plt.savefig(path+ folder + '\Results//ExpFitting_abs.png',dpi="figure",format="png",bbox_inches='tight')
    plt.clf()
    #plt.xscale('log')
    #plt.yscale('log')

    #plt.colorbar(colors)

    #plt.show()
    
    # Return desirable fitted parameters:
    return pars[1]

In [ ]:
# Plot the derivatives against each other.
def Dervs(energy_rates,strain_rates,times,path,folder):
    
    if folder == '2022_3_15_Anthony//1,2 F-0,24 M-2':
        index = 59
        # Plot the strain rates:
        temperature = times[0:59]
        #print(type(sum_energy[0:index]))
        colors = temperature 


        #print("Length of strain", len(strain_rates[0:59]))
        #print("Length of energy", len(energy_rates[0:59]))
        #print("Length of times", len(colors))
        plt.scatter(energy_rates[0:59],strain_rates[0:59],c=colors)
        plt.title('Cumulative Strain vs Cumulative Energy Consumption')
        # We always plot in J/L
        plt.xlabel('$\dot {e/V} \; (J/L/s )$',fontsize =20)
        plt.ylabel(r'$\dot \epsilon$ (1/s)',fontsize=20)
        plt.tick_params(axis='both', which='major', labelsize=tcksz)
        plt.savefig(path+ folder + '\Results//Derivs_2.png',dpi="figure",format="png")
        plt.clf()
        #plt.legend(loc="upper right")
        #plt.show()
    
    else:
        index = 88
        # Plot the strain rates:
        temperature = times[0:index]
        #print(type(sum_energy[0:index]))
        colors = temperature 


        plt.scatter(energy_rates[0:index],strain_rates[0:index],c=colors)
        plt.title('Cumulative Strain vs Cumulative Energy Consumption')
        # We always plot in J/L
        plt.xlabel('$\dot {e/V} \; (J/L/s )$',fontsize =20)
        plt.ylabel(r'$\dot \epsilon$ (1/s)',fontsize=20)
        plt.tick_params(axis='both', which='major', labelsize=tcksz)
        plt.savefig(path+ folder + '\Results//Derivs_2.png',dpi="figure",format="png")
        plt.clf()
        #plt.legend(loc="upper right")
        #plt.show()